# Notebook 04 — Carga, orquestación y buenas prácticas

Cuarto y último sub-bloque del Tema 04. Tomas los seis DataFrames del Notebook 03 y los **cargas** al `northwind_dwh` con `to_sql`. Después empaquetas todo el pipeline (extract + transform + load) en un script Python productivo con logging, manejo de errores y validaciones post-carga.

Cierra el tema con un panorama de cómo se ve un ETL **en producción real** — más allá del notebook.

Al terminar este notebook deberías tener el `northwind_dwh` poblado por tu ETL Python (intercambiable con los scripts SQL que ejecutaste en el Tema 01) y un `etl_pipeline.py` que puede correr de extremo a extremo.

## Setup

> *Por publicar.*

Re-ejecución de los notebooks anteriores (extract + clean + transform) para tener los 6 DataFrames listos.

In [ ]:
# TODO: re-ejecución encadenada hasta tener fact_sales, dim_customer, dim_product, dim_employee, dim_shipper, dim_date

## Carga al DWH con `to_sql`

> *Por publicar.*

Sintaxis y parámetros clave:

- `df.to_sql(name='dim_customer', con=engine, schema='northwind_dwh', if_exists='replace'|'append', index=False, chunksize=1000, dtype={...})`
- **`if_exists`:** `'replace'` (drop + create), `'append'` (insert), `'fail'` (error si existe).
- **`chunksize`:** divide el insert en lotes — performance + manejabilidad si algo falla.
- **`dtype`:** mapear columnas a tipos SQL específicos (ej. `NUMERIC(10,2)` para precios) en vez de dejar que pandas infiera.

In [ ]:
# TODO: cargar dim_customer con to_sql

## Estrategias: replace, append, upsert

> *Por publicar.*

- **`replace`** — drop + create + insert. Idempotente, simple. Útil para reload completo del DWH (caso del módulo).
- **`append`** — insertar sobre lo existente. **NO** es idempotente si reejecutas — duplica filas.
- **Upsert** (INSERT … ON CONFLICT) — pandas no lo soporta nativo; se hace con SQL crudo vía SQLAlchemy. Apropiado para cargas incrementales.

Conexión con el concepto de **idempotencia** del Notebook 01.

In [ ]:
# TODO: ejemplo de upsert con SQL crudo (INSERT ... ON CONFLICT)

## Tipos correctos al cargar — `dtype`

> *Por publicar.*

Sin `dtype` explícito, pandas + SQLAlchemy infieren los tipos SQL. La inferencia suele caer en `BIGINT` para enteros y `FLOAT` para decimales — **no es lo que quieres para precios**.

Patrón:

```python
from sqlalchemy.types import Integer, Numeric, String, Date, Boolean
df.to_sql(..., dtype={
    'unit_price': Numeric(10, 2),
    'product_name': String(40),
    'date_key': Integer(),
    'is_weekend': Boolean(),
})
```

Conexión con el `REAL → NUMERIC` que viste en el Tema 02: la decisión hay que aplicarla aquí también.

In [ ]:
# TODO: cargar fact_sales con dtype explícito para columnas críticas

## Orden de carga — importa por las FKs

> *Por publicar.*

Si tienes constraints de FK declaradas, **carga las dimensiones primero y la fact al final**. La fact apunta a las dims; cargarla antes haría que cada INSERT falle por FK no encontrada.

Orden recomendado: `dim_date`, `dim_customer`, `dim_product`, `dim_employee`, `dim_shipper`, después `fact_sales`.

In [ ]:
# TODO: carga ordenada de las 5 dims + fact_sales

## Pipeline modular — de notebook a `etl_pipeline.py`

> *Por publicar.*

El notebook es para **explorar y aprender**. Un ETL productivo vive en un **script `.py`**:

- Funciones separadas: `extract()`, `transform()`, `load()`.
- Orquestación con un `main()`.
- Parámetros vía CLI (`argparse`) o variables de entorno.
- Sin estado de notebook — todo determinístico desde la línea de comandos.

Plantilla mínima del script que se desarrollará a lo largo del notebook.

In [ ]:
# TODO: esqueleto de etl_pipeline.py
# def extract(engine): ...
# def transform(raw_data): ...
# def load(transformed_data, engine): ...
# def main(): ...
# if __name__ == '__main__': main()

## Logging básico

> *Por publicar.*

Módulo `logging` de la stdlib (NO `print`):

- Niveles: `DEBUG`, `INFO`, `WARNING`, `ERROR`, `CRITICAL`.
- Formato con timestamp + nivel + módulo + mensaje.
- Configuración con `logging.basicConfig(level=logging.INFO, format='...')`.
- Por qué importa: cuando el ETL corre en producción a las 3am sin tu supervisión, el log es **lo único** que tienes para diagnosticar.

In [ ]:
# TODO: configuración de logging + ejemplo de mensajes

## Manejo de errores y rollback

> *Por publicar.*

- `try/except` alrededor de los pasos críticos.
- **Transacciones SQLAlchemy** con `engine.begin()` para que el rollback sea automático si algo lanza excepción.
- Retry policies básicas (reintentar N veces con backoff) — mención conceptual, sin librería externa.
- Decisión clave: ¿reintentar o fallar rápido? Depende del tipo de error.

In [ ]:
# TODO: bloque transaccional con engine.begin() y manejo de excepciones

## Validaciones post-carga

> *Por publicar.*

Después de cargar, **verificar contra el origen**:

- Conteos: `SELECT count(*) FROM dwh.fact_sales` debe coincidir con `len(order_details)` del OLTP.
- Sumas y agregados: `SUM(line_total)` del DWH debe coincidir con `SUM(quantity * unit_price * (1-discount))` del OLTP.
- Spot-checks: una venta específica del OLTP debe aparecer correctamente en `fact_sales` con todas sus FKs resueltas.

Si alguna validación falla, **fallar el pipeline** — un DWH inconsistente es peor que ningún DWH.

In [ ]:
# TODO: queries de validación post-carga

## Cierre — ETL en producción real

> *Por publicar.*

Discusión conceptual, sin hands-on. Da contexto sobre dónde encaja lo aprendido:

- **Object storage (S3)** como zona de aterrizaje — los CSV / Parquet del día llegan a S3 antes de procesar.
- **Formatos columnares** (Parquet, ORC) — mucho más eficientes que CSV para datasets grandes.
- **Orquestación** (Airflow, Prefect, Dagster) — DAGs de tareas con dependencias, retries, alertas, scheduling.
- **Compute serverless** (AWS Glue, Lambda) — el ETL como función que escala sin servidores que mantener.
- **Tendencia ELT moderna** (dbt + Snowflake/BigQuery/Redshift) — transformar con SQL en el destino columnar masivo.

Lo que aprendiste en este tema es la **lógica fundamental** que sigue siendo la misma; solo cambian las herramientas que la orquestan.

## Y con esto cierras el Tema 04

> *Por publicar.*

Resumen de lo construido en los cuatro notebooks: un ETL completo en Python que toma `northwind_oltp`, lo transforma al modelo dimensional y pobla `northwind_dwh`. Conexión con lo que viene en el Tema 05 (SQL avanzado) — ahora que el DWH está cargado, las queries analíticas son el siguiente nivel.

---

<p align="center">
<a href="03_transformacion.ipynb">← Anterior: Notebook 03</a> | <a href="Readme.md">Volver al índice del Tema 04</a> | <a href="../Tema-05/Readme.md">Siguiente: Tema 05 →</a>
</p>